Эта часть кода не предназначена для запуска на устройстве, её стоит запускать на Kaggle при включенном GPU

In [ ]:
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

MODEL_NAME = "allenai/specter2_base"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device).eval()

# ---------- 3. Кодирование ----------
def mean_pool(last_hidden, attention_mask):
    mask = attention_mask.unsqueeze(-1).float()
    return (last_hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

@torch.no_grad()
def encode(texts, batch_size=64, max_len=512):
    embs = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i + batch_size]
        enc = tokenizer(
            batch, padding=True, truncation=True,
            max_length=max_len, return_tensors="pt"
        ).to(device)
        out = model(**enc).last_hidden_state
        emb = mean_pool(out, enc["attention_mask"])
        emb = torch.nn.functional.normalize(emb, dim=-1)
        embs.append(emb.cpu().numpy())
    return np.vstack(embs)



import json

doc_embs = np.load("doc_embs.npy")
with open("ids.json", encoding="utf-8") as f:
    ids = json.load(f)
with open("meta.json", encoding="utf-8") as f:
    meta = json.load(f)

# быстрый доступ id -> title
title_by_id = {m["id"]: m["title"] for m in meta}

def search(query, top_k=10):
    q = encode([query], batch_size=1)[0]      # encode должен быть определён
    scores = doc_embs @ q                      # косинус, эмбеддинги нормированы
    top = np.argpartition(-scores, top_k)[:top_k]
    top = top[np.argsort(-scores[top])]
    return [
        {
            "id": ids[i],
            "title": title_by_id.get(ids[i], "—"),
            "score": float(scores[i]),
        }
        for i in top
    ]

def pretty_print(query, top_k=10):
    print(f"\n🔎 {query}")
    for r in search(query, top_k=top_k):
        print(f"  {r['score']:.3f}  [{r['id']}]  {r['title']}")


while True:
    query = input("Ask your question:")
    if query in ["exit", "q", "quit", "end"]:
        break
    top_k = int(input("How much answers you want?:"))
    pretty_print(query, top_k)